# Day 4.4 — Parallel Specialist Reviewers


## Before you begin

### Learning outcomes

- Name the four pieces of a fan-out precisely: shared state, fan-out, handoff, fan-in.
- Run three narrow reviewers sequentially, then in threads, and time both.
- Show that parallelism buys wall-clock time and nothing else.

Architecture reference: [Day 4 diagrams D13](../../diagrams/source/day_04.md)

### Expected observation

Three specialists each report only their own category; the sequential run takes about 0.9 s and the threaded run about 0.3 s, with identical calls and tokens.


## Concept briefing

## Specialist decomposition

A specialist role must narrow the task, not merely rename the same prompt. Our
correctness, security and maintainability reviewers all read the same immutable artifact
but answer different questions, and all return the same `Finding` contract: category,
location, evidence, severity, recommendation, and which role produced it.

That contract is the **handoff**. It is what crosses the boundary between agents - not
personas, not hidden reasoning, not a full chat transcript. The supervisor does not need
each reviewer's conversation; it needs validated records with enough provenance to resolve
duplicates and conflicts.

Narrowing a prompt does not create knowledge the reviewer never had. If the generalist
cannot see a subtle business rule, the specialist with a narrower prompt usually cannot
see it either. Decomposition redistributes attention; it does not add capability.

## Sequential before parallel

Say the words precisely:

- **Fan-out**: one step launches several independent branches.
- **Shared state**: what all the branches read. Here it is the immutable artifact text.
  Nothing mutates it, which is exactly why the branches are safe to run at the same time.
- **Fan-in**: one step collects every branch's results and combines them.

Run the branches sequentially first, because the order and any failure are easy to read.
Then, if the branches truly do not depend on one another, run them in threads. Wall-clock
time drops because the calls wait on the network together. The number of calls, the tokens
and the bill do not drop at all - and parallel calls hit provider rate limits sooner.

The fan-in step must be bounded: validate fields, merge duplicates, rank, cap the output,
stop. A supervisor that can keep asking for revisions has become another autonomous loop
rather than a controlled aggregation step - and it must report what it merged and what it
truncated, or the cap silently deletes findings.


In [ ]:
# --- Course setup: run this cell first -----------------------------------------
# 1) Locate this day's folder so we can import from src/ and read data/ no matter
#    where Jupyter, VS Code, or Colab started. Every file path below goes through
#    PROJECT_ROOT, never through the current working directory.
import os, sys
from pathlib import Path

def find_project_root(marker="src/review_team"):
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in [folder, *folder.glob("day_*")]:
            if (candidate / marker).exists():
                return candidate
    raise FileNotFoundError(
        "Course folder not found. On Google Colab run the 'Colab bootstrap' cell at the top "
        "of the day notebook first; locally, start Jupyter inside the repository folder."
    )

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 2) Load the API key from the .env file at the repository root (Day 1.1 shows how to
#    create it). If no key is present we stay in deterministic MOCK mode: every cell
#    still runs, answers are fixed strings, and no credit is spent.
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))
LIVE = bool(os.getenv("OPENROUTER_API_KEY"))

print("Project root :", PROJECT_ROOT)
print("Mode         :", "LIVE (OpenRouter)" if LIVE else "MOCK (no OPENROUTER_API_KEY found)")


## Step 1 — Load the artifact


In [ ]:
# The artifact under review and the instructor's answer key.
# Both are plain files; nothing here is secret from you, but the answer key is never
# put into a reviewer's prompt.
ARTIFACT_PATH = PROJECT_ROOT / "data" / "seeded_artifact" / "order_service.py"
GOLDEN_PATH   = PROJECT_ROOT / "data" / "golden_defects.json"

SOURCE = ARTIFACT_PATH.read_text(encoding="utf-8")   # the shared, immutable artifact

print("Artifact file :", ARTIFACT_PATH.name)
print("Artifact lines:", len(SOURCE.splitlines()))
print("Answer key    :", GOLDEN_PATH.name)


## Step 2 — The vocabulary, on this exact example

- **Shared state** — what every branch reads. Here it is `SOURCE`, one immutable string. Nothing writes to it, which is precisely why the branches are safe to run at the same time.
- **Fan-out** — one step launches several independent branches. Ours is bounded at three: three roles, three calls, no branch may spawn another.
- **Handoff** — what crosses the boundary between agents. Only `Finding` records: no personas, no chat history, no hidden reasoning.
- **Fan-in** — one step collects every branch's output and combines it. That is the supervisor, and it is the whole of the next notebook.


In [ ]:
from review_team import SPECIALIST_ROLES

print("Bounded fan-out width:", len(SPECIALIST_ROLES))
print("Roles                :", list(SPECIALIST_ROLES))
print()
print()
print("Shared state: one immutable string that every branch reads.")
print("   artifact length (characters):", len(SOURCE))
print("   any branch writing to it? no - each returns its own list of findings")


## Step 3 — What a narrow role actually changes

A specialist is not the same prompt with a new name. Each one answers a different question about the *same* text, and is only allowed to report findings in its own category.


In [ ]:
from review_team import specialist_review

for role in SPECIALIST_ROLES:
    findings = specialist_review(SOURCE, role)
    print(f"\n{role} specialist -> {len(findings)} finding(s)")
    for finding in findings:
        print(f"   line {finding.line:>3} | {finding.severity:<8} | {finding.title}")
    # The role contract: a specialist may not stray outside its category.
    assert all(f.category == role for f in findings)
print("\nEvery finding stayed inside its reviewer's category.")


## Step 4 — Look at one handoff record

This is the entire message the security specialist sends onward. Compare it with "here is my conversation, please read it": it is small, validated, and mergeable.


In [ ]:
handoff = specialist_review(SOURCE, "security")[0]

for field_name, value in handoff.as_dict().items():
    print(f"{field_name:>15}: {value}")
print("\nWhat is NOT in the handoff: the prompt, the reasoning, the chat history.")


## Step 5 — Sequential first, because it is easy to debug

We give the mock reviewer a 0.3 second delay so it behaves like a real network call. Sequential means: call one, wait, call the next, wait, call the last.


In [ ]:
from time import perf_counter
from review_team import MockStructuredReviewer, run_specialist_team

# A reviewer that pretends each call takes 0.3 s of network time.
slow_provider = MockStructuredReviewer("blind_spots", latency_s=0.3)

start = perf_counter()
sequential = run_specialist_team(SOURCE, slow_provider, parallel=False)
sequential_seconds = perf_counter() - start

print("Execution   : sequential")
print("Model calls : %d" % sequential.model_calls)
print("Tokens      : %d" % sequential.total_tokens)
print("Wall clock  : %.2f s" % sequential_seconds)


## Step 6 — Now fan out into threads

`ThreadPoolExecutor` starts the three calls together. While one branch is waiting on the network, the others are waiting too — so the total wait is roughly the *longest* call, not the sum of all three.


In [ ]:
start = perf_counter()
parallel = run_specialist_team(SOURCE, slow_provider, parallel=True)
parallel_seconds = perf_counter() - start

print("Execution   : parallel (ThreadPoolExecutor, max_workers=3)")
print("Model calls : %d" % parallel.model_calls)
print("Tokens      : %d" % parallel.total_tokens)
print("Wall clock  : %.2f s" % parallel_seconds)


## Step 7 — What parallelism did and did not buy

Read the table carefully. One column improves. The two columns you pay money for do not move at all.


In [ ]:
print(f"{'measure':<22}{'sequential':>12}{'parallel':>12}")
print("-" * 46)
print(f"{'wall clock (s)':<22}{sequential_seconds:>12.2f}{parallel_seconds:>12.2f}")
print(f"{'model calls':<22}{sequential.model_calls:>12}{parallel.model_calls:>12}")
print(f"{'tokens':<22}{sequential.total_tokens:>12}{parallel.total_tokens:>12}")
print(f"{'findings reported':<22}{len(sequential.findings):>12}{len(parallel.findings):>12}")
print()
print("Speed-up: %.1fx" % (sequential_seconds / parallel_seconds))
print("Same findings in the same order:",
      [f.id for f in sequential.findings] == [f.id for f in parallel.findings])
print()
print("Parallel execution buys wall-clock time. It does not reduce calls, tokens or cost,")
print("and it reaches a provider's rate limit three times faster.")


### Try it yourself

The pool is capped at 3 workers. Predict the wall clock if we capped it at **1** worker instead, keeping everything else identical.


In [ ]:
# --- Worked solution ---
# Fan out by hand so the bound is visible: max_workers is the whole cap.
from concurrent.futures import ThreadPoolExecutor

def timed_fan_out(max_workers):
    start = perf_counter()
    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        # Each branch is one call to one specialist; all read the same shared SOURCE.
        results = list(pool.map(lambda role: slow_provider.review(SOURCE, role),
                                SPECIALIST_ROLES))
    seconds = perf_counter() - start
    total_findings = sum(len(findings) for findings, _usage in results)
    return seconds, total_findings

for workers in (1, 2, 3):
    seconds, total = timed_fan_out(workers)
    print("max_workers=%d -> %.2f s, %d findings, 3 model calls" % (workers, seconds, total))

print()
print("max_workers=1 is sequential execution wearing a thread pool.")
print("The findings and the bill never change; only how long we wait does.")


### Checkpoint

**1. Why is it safe to run these three reviewers at the same time?**

<details><summary>Show answer</summary>

Because the shared state is read-only. Every branch reads the same immutable `SOURCE` string and writes only to its own list of findings, so there is no order in which they could interfere. The moment a branch needed to *modify* shared state, we would need locking or a merge rule, and the simple fan-out would stop being safe.

</details>

**2. Your manager says "run the specialists in parallel to cut our API bill". What do you reply?**

<details><summary>Show answer</summary>

Parallelism does not touch the bill. We measured it: sequential and threaded runs made the same 3 calls and used the same tokens, and only the wall clock fell (about 0.9 s to 0.3 s). To cut cost you must remove calls — for example by proving findings with the AST checker or by using one reviewer instead of three.

</details>

### Recap

- Limitation we saw: running three specialists one after another takes the sum of all three waits.
- Layer we added: a bounded fan-out over read-only shared state, with `Finding` records as the only handoff.
- Evidence it worked: 0.9 s -> 0.3 s wall clock, identical findings, identical 3 calls and identical tokens.
